# Extract Google ClusterData2019 + PowerData2019

This notebook extracts:
1. Per-cell CPU utilization (5-min intervals) for cells a-d
2. Power model calibration data (CPU vs power utilization)

**Before running:** You need a GCP project with BigQuery API enabled.
- Go to https://console.cloud.google.com
- Create a project if you don't have one
- Enable BigQuery API: https://console.cloud.google.com/flows/enableapi?apiid=bigquery

In [ ]:
# Step 1: Authenticate and set project ID
from google.colab import auth
auth.authenticate_user()

# CHANGE THIS to your GCP project ID
PROJECT_ID = 'YOUR_PROJECT_ID_HERE'  # <-- EDIT THIS

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)
print(f'Authenticated with project: {PROJECT_ID}')

In [ ]:
# Step 2: Test connection - query cell 'a' capacity
test_query = """
SELECT SUM(cpu_cap) AS cpu_capacity
FROM (
    SELECT machine_id, MAX(capacity.cpus) AS cpu_cap
    FROM `google.com:google-cluster-data`.clusterdata_2019_a.machine_events
    GROUP BY 1
)
"""
result = client.query(test_query).to_dataframe()
print(f"Cell 'a' CPU capacity: {result['cpu_capacity'].iloc[0]:.2f}")
print('Connection works!')

## Extract Cell Workloads
Extracts 5-minute aggregate CPU utilization per cell, normalized by cell capacity to [0, 1].

In [ ]:
import pandas as pd
import numpy as np
import os

os.makedirs('data/cells', exist_ok=True)

CELLS = ['a', 'b', 'c', 'd']

def get_cell_capacity(cell):
    query = f"""
    SELECT SUM(cpu_cap) AS cpu_capacity
    FROM (
        SELECT machine_id, MAX(capacity.cpus) AS cpu_cap
        FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.machine_events
        GROUP BY 1
    )
    """
    result = client.query(query).to_dataframe()
    return float(result['cpu_capacity'].iloc[0])


def extract_cell_utilization(cell, cpu_capacity):
    query = f"""
    SELECT
        CAST(FLOOR(start_time / (1e6 * 300)) AS INT64) AS time_bucket,
        SUM(average_usage.cpus) / {cpu_capacity} AS cpu_demand_norm
    FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.instance_usage
    WHERE (alloc_collection_id IS NULL OR alloc_collection_id = 0)
        AND (end_time - start_time) >= (5 * 60 * 1e6)
    GROUP BY 1
    ORDER BY 1
    """
    df = client.query(query).to_dataframe()
    df['timestep'] = df['time_bucket'] - df['time_bucket'].min()
    df = df[['timestep', 'cpu_demand_norm']].copy()
    df['cpu_demand_norm'] = df['cpu_demand_norm'].clip(0.0, 1.0)
    return df


for cell in CELLS:
    print(f'\n--- Cell {cell} ---')
    cap = get_cell_capacity(cell)
    print(f'  CPU capacity: {cap:.2f}')
    
    print(f'  Extracting 5-min utilization (this may take a few minutes)...')
    df = extract_cell_utilization(cell, cap)
    
    path = f'data/cells/cell_{cell}.csv'
    df.to_csv(path, index=False)
    print(f'  Saved {len(df)} rows to {path}')
    print(f'  Mean={df["cpu_demand_norm"].mean():.4f}, Max={df["cpu_demand_norm"].max():.4f}')

print('\nDone extracting cell workloads!')

## Extract Power Model Data
Extracts hourly CPU utilization vs measured power utilization, fits a linear model.

In [ ]:
import json

os.makedirs('data', exist_ok=True)

# Extract hourly power utilization per cell
print('Extracting hourly power utilization...')
power_query = """
SELECT
    cell,
    CAST(FLOOR(time / (1e6 * 60 * 60)) AS INT64) AS hour_index,
    AVG(measured_power_util) AS avg_power_util
FROM `google.com:google-cluster-data`.`powerdata_2019.cell*`
WHERE NOT bad_measurement_data
    AND cell IN ('a', 'b', 'c', 'd')
GROUP BY 1, 2
ORDER BY 1, 2
"""
power_df = client.query(power_query).to_dataframe()
print(f'  Got {len(power_df)} power rows')

# Extract hourly CPU utilization per cell
all_cpu = []
for cell in CELLS:
    print(f'Extracting hourly CPU for cell {cell}...')
    cap = get_cell_capacity(cell)
    
    cpu_query = f"""
    SELECT
        CAST(FLOOR(start_time / (1e6 * 60 * 60)) AS INT64) AS hour_index,
        SUM(average_usage.cpus) / (12 * {cap}) AS avg_cpu_util
    FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.instance_usage
    WHERE (alloc_collection_id IS NULL OR alloc_collection_id = 0)
        AND (end_time - start_time) >= (5 * 60 * 1e6)
    GROUP BY 1
    ORDER BY 1
    """
    cpu_df = client.query(cpu_query).to_dataframe()
    cpu_df['cell'] = cell
    all_cpu.append(cpu_df)
    print(f'  Got {len(cpu_df)} rows')

cpu_combined = pd.concat(all_cpu, ignore_index=True)

# Merge on (cell, hour_index)
merged = pd.merge(cpu_combined, power_df, on=['cell', 'hour_index'], how='inner')
print(f'\nMerged dataset: {len(merged)} rows')

# Filter valid rows
cpu_util = merged['avg_cpu_util'].values
power_util = merged['avg_power_util'].values
valid = np.isfinite(cpu_util) & np.isfinite(power_util)
cpu_util = cpu_util[valid]
power_util = power_util[valid]

# Fit linear model: P = intercept + slope * cpu
A = np.vstack([np.ones_like(cpu_util), cpu_util]).T
result = np.linalg.lstsq(A, power_util, rcond=None)
intercept, slope = result[0]

predicted = intercept + slope * cpu_util
ss_res = np.sum((power_util - predicted) ** 2)
ss_tot = np.sum((power_util - power_util.mean()) ** 2)
r_squared = 1.0 - ss_res / ss_tot

params = {
    'idle_power': float(intercept),
    'peak_power': float(intercept + slope),
    'slope': float(slope),
    'r_squared': float(r_squared),
    'description': 'Linear power model: P = idle_power + slope * cpu_utilization'
}

print(f'\nFitted power model:')
print(f'  P_idle  = {params["idle_power"]:.4f}')
print(f'  P_peak  = {params["peak_power"]:.4f}')
print(f'  slope   = {params["slope"]:.4f}')
print(f'  R^2     = {params["r_squared"]:.4f}')

with open('data/power_model_params.json', 'w') as f:
    json.dump(params, f, indent=2)
print('Saved data/power_model_params.json')

scatter_df = pd.DataFrame({'cpu_util': cpu_util, 'power_util': power_util})
scatter_df.to_csv('data/power_model_scatter.csv', index=False)
print('Saved data/power_model_scatter.csv')

In [ ]:
# Quick visualization of the power model fit
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(cpu_util, power_util, alpha=0.1, s=2, label='Data')
x_line = np.linspace(0, cpu_util.max(), 100)
y_line = intercept + slope * x_line
ax.plot(x_line, y_line, 'r-', linewidth=2,
        label=f'P = {intercept:.3f} + {slope:.3f} * CPU (R²={r_squared:.3f})')
ax.set_xlabel('CPU Utilization')
ax.set_ylabel('Power Utilization')
ax.set_title('Power Model: CPU vs Power (Google PowerData2019)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Download Files
Run this cell to download all extracted data as a zip file.

In [ ]:
import shutil
from google.colab import files

# Create zip of the data directory
shutil.make_archive('extracted_data', 'zip', '.', 'data')
files.download('extracted_data.zip')
print('Download started! Extract into C:\\Projects\\thesis\\ on your local machine.')